In [0]:
import os
import json
import urllib.parse
import urllib.request
from delta.tables import DeltaTable
from datetime import date, timedelta
from pyspark.sql.window import Window
from pyspark.sql import functions as F



In [0]:
data_fim = date.today()
data_inicio = data_fim 

data_inicio_api = data_inicio.strftime("%m-%d-%Y")
data_fim_api = data_fim.strftime("%m-%d-%Y")

data_inicio_ref = data_inicio.strftime("%Y-%m-%d")
data_fim_ref = data_fim.strftime("%Y-%m-%d")

batch_id = f"{data_inicio_ref}_{data_fim_ref}".replace("-", "")

In [0]:
catalogo = "databricks_cata_managed"
df_bronze = spark.table(f"{catalogo}.bronze.cambio_ptax_raw").filter(F.col("batch_id") == batch_id)

In [0]:
df_stage_raw = (
    df_bronze
    .select(
        F.col("moeda").alias("codigo_moeda"),
        F.col("batch_id"),
        F.col("_arquivo_lido"),
        F.col("_data_ingestao"),
        F.explode_outer(F.col("registros")).alias("registro")
    )
    .select(
        F.col("codigo_moeda"),
        F.to_date(F.col("registro.dataHoraCotacao")).alias("data_cotacao"),
        F.to_timestamp(F.col("registro.dataHoraCotacao")).alias("data_hora_cotacao"),

        F.col("registro.cotacaoCompra").cast("decimal(18,6)").alias("cotacao_compra"),
        F.col("registro.cotacaoVenda").cast("decimal(18,6)").alias("cotacao_venda"),
        F.col("registro.tipoBoletim").cast("string").alias("tipo_boletim"),

        F.col("batch_id"),
        F.col("_arquivo_lido"),
        F.col("_data_ingestao")
    )
    .filter(F.col("data_hora_cotacao").isNotNull())
)

In [0]:
w = (
    Window
    .partitionBy("codigo_moeda", "data_hora_cotacao", "tipo_boletim")
    .orderBy(F.col("_data_ingestao").desc())
)

df_stage = (
    df_stage_raw
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.silver.cotacao_moeda (
    codigo_moeda STRING,
    data_cotacao DATE,
    data_hora_cotacao TIMESTAMP,
    cotacao_compra DECIMAL(18,6),
    cotacao_venda DECIMAL(18,6),
    tipo_boletim STRING,
    batch_id STRING,
    _arquivo_lido STRING,
    _data_ingestao TIMESTAMP,
    _data_atualizacao TIMESTAMP
)
USING DELTA
""")

In [0]:
tabela_destino = f"{catalogo}.silver.cotacao_moeda"

delta_silver = DeltaTable.forName(spark, tabela_destino)

(
    delta_silver.alias("destino")
    .merge(
        df_stage.alias("origem"),
        """
        destino.codigo_moeda = origem.codigo_moeda
        AND destino.data_hora_cotacao = origem.data_hora_cotacao
        AND destino.tipo_boletim = origem.tipo_boletim
        """
    )
    .whenMatchedUpdate(set={
        "data_cotacao": "origem.data_cotacao",
        "cotacao_compra": "origem.cotacao_compra",
        "cotacao_venda": "origem.cotacao_venda",
        "batch_id": "origem.batch_id",
        "_arquivo_lido": "origem._arquivo_lido",
        "_data_ingestao": "origem._data_ingestao",
        "_data_atualizacao": "current_timestamp()"
    })
    .whenNotMatchedInsert(values={
        "codigo_moeda": "origem.codigo_moeda",
        "data_cotacao": "origem.data_cotacao",
        "data_hora_cotacao": "origem.data_hora_cotacao",
        "cotacao_compra": "origem.cotacao_compra",
        "cotacao_venda": "origem.cotacao_venda",
        "tipo_boletim": "origem.tipo_boletim",
        "batch_id": "origem.batch_id",
        "_arquivo_lido": "origem._arquivo_lido",
        "_data_ingestao": "origem._data_ingestao",
        "_data_atualizacao": "current_timestamp()"
    })
    .execute()
)